In [34]:
# ================================================================
#             📌 BIODIVERSITY PROCESSING FULL PIPELINE
# ================================================================

# ---  Préambule : imports ---
%matplotlib inline

import os, sys, importlib
from pathlib import Path
import ipynbname
import geopandas as gpd
import pandas as pd

# ================================================================
# 2️⃣ GLOBAL PATH CONFIGURATION
# ================================================================
base_path = Path(ipynbname.path()).parent.parent.parent
path_data = base_path / 'Data'
scripts_path = base_path / 'Code' / 'Formatage' / 'scripts'
sys.path.append(str(scripts_path.resolve()))

print("📌 **CONFIGURATION DES CHEMINS**")
print(f"📍 base_path    : {base_path}")
print(f"📂 path_data     : {path_data}")
print(f"📂 scripts_path  : {scripts_path}")

# ================================================================
# 3️⃣ IMPORT CUSTOM MODULES
# ================================================================
import formatage_geo, formatage_biodiv, formatage_ObsToGrid, formatage_fusion
importlib.reload(formatage_geo)
importlib.reload(formatage_biodiv)
importlib.reload(formatage_ObsToGrid)
importlib.reload(formatage_fusion)

from formatage_geo import generate_country_grid, plot_country_grid, simplify_world_geometry, save_country_grid
from formatage_biodiv import read_biodiv_chunks, save_clean_biodiv,inspect_columns_with_examples, detect_sep,harmonize_columns,clean_biodiv_data_source
from formatage_ObsToGrid import assign_grid_to_points, generate_taxo_dict, aggregate_by_grid_species, save_processed_biodiv
from formatage_fusion import fusion_cells_by_obs, apply_fusion_to_biodiv, plot_fusionned_grid, save_fusionned_biodiv,save_merged_grid


📌 **CONFIGURATION DES CHEMINS**
📍 base_path    : C:\Users\Aubin\Documents\MANTIS
📂 path_data     : C:\Users\Aubin\Documents\MANTIS\Data
📂 scripts_path  : C:\Users\Aubin\Documents\MANTIS\Code\Formatage\scripts


In [6]:
# --- Lecture Plantae ---
path_fichier = path_data/'SILENE'/'raw'/'SILENE_MartiguesEtAlentours_Plantae.csv'
sep = detect_sep(path_fichier)
df_Plantae = pd.read_csv(path_fichier, sep=sep, on_bad_lines="skip")
df_Plantae["regne"] = "Plantae"   # 💡 ajout du règne

# --- Lecture Arthropoda ---
path_fichier = path_data/'SILENE'/'raw'/'SILENE_MartiguesEtAlentours_Arthropoda.csv'
df_Arth = pd.read_csv(path_fichier, sep=sep, on_bad_lines="skip")
df_Arth["regne"] = "Animalia"     # 💡 ajout du règne


✅ Séparateur choisi : ';' avec 32 colonnes


In [31]:
import pandas as pd

# --- Concaténation ---
df_silene = pd.concat([df_Plantae, df_Arth], ignore_index=True)

# --- Vérifier les colonnes ---
print("Colonnes finales :", df_silene.columns.tolist())

# --- Enregistrement ---
path_sortie = path_data / 'SILENE' /'raw'/ 'SILENE_Martigues.csv'
df_silene.to_csv(path_sortie, index=False)
print(f"✅ Fichier concaténé enregistré : {path_sortie}")


Colonnes finales : ['uuid_perm_sinp', 'uuid_perm_grp_sinp', 'jdd_nom', 'jdd_uuid', 'fournisseur', 'observateurs', 'cd_ref', 'nom_valide', 'nom_vernaculaire', 'classe', 'famille', 'ordre', 'nombre_min', 'nombre_max', 'date_debut', 'date_fin', 'geojson_4326', 'x_centroid_4326', 'y_centroid_4326', 'precision_geographique', 'type_precision', 'communes', 'alti_min', 'technique_observation', 'stade_vie', 'statut_biologique', 'sexe', 'comportement', 'type_source', 'sensibilite', 'confidentialite', 'floutage', 'regne']
✅ Fichier concaténé enregistré : C:\Users\Aubin\Documents\MANTIS\Data\SILENE\raw\SILENE_Martigues.csv


In [21]:
df_silene

,uuid_perm_sinp,uuid_perm_grp_sinp,jdd_nom,jdd_uuid,fournisseur,observateurs,cd_ref,nom_valide,nom_vernaculaire,classe,...,technique_observation,stade_vie,statut_biologique,sexe,comportement,type_source,sensibilite,confidentialite,floutage,regne
0,5631821f-8d83-11e7-885f-1cc1dee9999c,120edd69-8d7c-11e7-885f-1cc1dee9999c,Divers jeux de données floristiques produits p...,be81c44f-b8e7-464f-959d-4027b084dd96,Conservatoire botanique national méditerranéen,MICHAUD Henri (CBNMed),110473,"Ophrys speculum Link, 1800","Ophrys miroir, Ophrys cilié",Equisetopsida,...,Vu,Inconnu,Non renseigné,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Plantae
1,5631ae6d-8d83-11e7-885f-1cc1dee9999c,120f0c50-8d7c-11e7-885f-1cc1dee9999c,Divers jeux de données floristiques produits p...,be81c44f-b8e7-464f-959d-4027b084dd96,Conservatoire botanique national méditerranéen,MICHAUD Henri (CBNMed),92139,"Colchicum filifolium (Cambess.) Stef., 1926","Colchique à feuilles filiformes, Mérendère à f...",Equisetopsida,...,Vu,Inconnu,Non renseigné,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Plantae
2,5632726e-8d83-11e7-885f-1cc1dee9999c,120fb50c-8d7c-11e7-885f-1cc1dee9999c,Divers jeux de données floristiques produits p...,be81c44f-b8e7-464f-959d-4027b084dd96,Conservatoire botanique national méditerranéen,LEOTARD Guillaume (CBNMed),127915,"Tulipa agenensis DC., 1804","Tulipe d'Agen, Tulipe oeil-de-soleil",Equisetopsida,...,Vu,Inconnu,Non renseigné,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Plantae
3,563277e9-8d83-11e7-885f-1cc1dee9999c,120fbbc6-8d7c-11e7-885f-1cc1dee9999c,Divers jeux de données floristiques produits p...,be81c44f-b8e7-464f-959d-4027b084dd96,Conservatoire botanique national méditerranéen,BLASCO André (CBNMed),100543,"Gomphocarpus fruticosus (L.) W.T.Aiton, 1811","Petite ouate, Ouatier marron",Equisetopsida,...,Vu,Inconnu,Non renseigné,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Plantae
4,56328e29-8d83-11e7-885f-1cc1dee9999c,120fd6be-8d7c-11e7-885f-1cc1dee9999c,Divers jeux de données floristiques produits p...,be81c44f-b8e7-464f-959d-4027b084dd96,Conservatoire botanique national méditerranéen,MICHAUD Henri (CBNMed),113230,"Phleum subulatum (Savi) Asch. & Graebn., 1899",Fléole subulée,Equisetopsida,...,Vu,Inconnu,Non renseigné,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Plantae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121961,9c9856ba-34b0-49c2-a311-eb6c6d5d7569,4e61f65b-a278-435f-a8bb-ae838c6530c4,CEN PACA_Lumière attractive Lepiled,0e4ab11e-a8dd-47bf-996c-afd9c90d2d8f,Conservatoire d'espaces naturels Provence-Alpe...,"BENCE Stéphane, ROLLAND Robin",249410,"Spodoptera exigua (Hübner, 1808)",Noctuelle exiguë (La),Insecta,...,Inconnu,Imago,NaN,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Animalia
121962,e3c29a01-be91-43f8-b207-510e6eb4cbc0,4e61f65b-a278-435f-a8bb-ae838c6530c4,CEN PACA_Lumière attractive Lepiled,0e4ab11e-a8dd-47bf-996c-afd9c90d2d8f,Conservatoire d'espaces naturels Provence-Alpe...,"BENCE Stéphane, ROLLAND Robin",249200,"Agrotis ipsilon (Hufnagel, 1766)",Noctuelle baignée (La),Insecta,...,Inconnu,Imago,NaN,Non renseigné,Inconnu,Terrain,donnée non sensible,donnée non confidentielle,NON,Animalia
121963,ca256525-414e-4729-a171-28cffa6f8539,666108b4-6db1-42de-8275-0723e1f6fd20,2020_ECO-MED_4,f3534706-7072-4788-85ba-3308df3df1e9,Bureau d'études ECO-MED,IORIO Etienne,233234,"Singa nitidula C.L. Koch, 1844",NaN,Arachnida,...,Vu,Indéterminé,NaN,Indéterminé,Non renseigné,Terrain,donnée non sensible,donnée non confidentielle,NON,Animalia
121964,44be355d-74d6-13cf-e053-2614a8c01513,778205a5-9196-450e-af6f-7267408c0301,2020_MNHN_1,95058f26-6ed8-4e52-9cc9-dfa07e2462fa,Muséum national d'Histoire naturelle,PONEL Philippe,237085,"Porcellio dilatatus Brandt, 1833",NaN,Malacostraca,...,Inconnu,Inconnu,Na

In [35]:
# biodiv_utils.py

import pandas as pd
import os, csv
from pathlib import Path
import numpy as np
import importlib

import formatage_biodiv_config
importlib.reload(formatage_biodiv_config)
from formatage_biodiv_config import colonnes_import, col_map

source='SILENE'
colonnes_a_importer = colonnes_import.get(source)
if colonnes_a_importer is None:
        raise ValueError(f"Source {source} non reconnue dans colonnes_import")

sep = detect_sep(path_sortie)
df_final = pd.DataFrame()
chunk_number = 0

df=pd.read_csv(
    path_sortie,
    sep=sep,
    usecols=lambda c: c in colonnes_a_importer,
    on_bad_lines='skip')

print(f"\n--- Traitement chunk {chunk_number} ---")
df_har = harmonize_columns(df, source)
#df_clean = clean_biodiv_data(df_chunk, cle_ID=cle_ID, annee_min=annee_min)
df_clean = clean_biodiv_data_source(df_har, source=source)
df_final = pd.concat([df_final, df_clean], ignore_index=True)
print(f"✅ Chunk {chunk_number} traité. Total lignes cumulées : {len(df_final)}")

✅ Séparateur choisi : ',' avec 33 colonnes

--- Traitement chunk 0 ---
   🔧 Dates (année + mois valides)   -17 lignes (0.01%)
   🔧 clean coords                   -0 lignes (0.0%)
   🔧 clean ID                       -0 lignes (0.0%)
   🔧 clean nombreObs                -0 lignes (0.0%)
✅ Chunk 0 traité. Total lignes cumulées : 121949


In [36]:
df_final

,speciesID,nombreObs,species,genus,family,order,class,kingdom,month,year,lon,lat
0,110473,1,Ophrys speculum,Ophrys,Orchidaceae,Asparagales,Equisetopsida,Plantae,4,1999,5.167290,43.399400
1,92139,1,Colchicum filifolium,Colchicum,Colchicaceae,Liliales,Equisetopsida,Plantae,10,2001,5.025407,43.341259
2,127915,1,Tulipa agenensis,Tulipa,Liliaceae,Liliales,Equisetopsida,Plantae,3,2001,5.035244,43.368773
3,100543,1,Gomphocarpus fruticosus,Gomphocarpus,Apocynaceae,Gentianales,Equisetopsida,Plantae,11,2000,4.948828,43.439999
4,113230,1,Phleum subulatum,Phleum,Poaceae,Poales,Equisetopsida,Plantae,8,2000,5.119634,43.395663
...,...,...,...,...,...,...,...,...,...,...,...,...
121944,249410,1,Spodoptera exigua,Spodoptera,Noctuidae,Lepidoptera,Insecta,Animalia,8,2025,5.065120,43.351928
121945,249200,1,Agrotis ipsilon,Agrotis,Noctuidae,Lepidoptera,Insecta,Animalia,8,2025,5.065120,43.351928
121946,233234,1,Singa nitidula,Singa,Araneidae,Araneae,Arachnida,Animalia,7,2012,5.006634,43.443021
121947,237085,1,Porcellio dilatatus,Porcellio,Porcellionidae,Isopoda,Malacostraca,Animalia,12,2015,4.883198,43.445236
